# Transformer の翻訳モデルをゼロから学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_machine_translation/nmt_from_scratch.ipynb)

NLP クックブック 23-09 ／ 機械翻訳

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

対訳データだけから、ニューラル機械翻訳モデルを自分で学習したい。サブワード分割、モデルの定義、学習、ビームサーチ、BLEU による評価までを一通り行う。

In [ ]:
!pip install -q sentencepiece sacrebleu

In [ ]:
import gzip
import random
import re
import urllib.request

# 田中コーパス (EDRDG 配布版、CC BY)。Tatoeba の日英対訳で、"A: 日本語<TAB>English#ID=..." の行が 1 対
raw = gzip.decompress(urllib.request.urlopen("https://www.edrdg.org/pub/Nihongo/examples.utf.gz").read()).decode("utf-8")
pairs = [m.groups() for m in re.finditer(r"^A: (.+?)\t(.+?)#ID=", raw, re.M)]

# 日本語・英語のどちらかが同じ対は 1 つだけ残す (学習とテストに同じ文が入るのを防ぐ)
seen_ja, seen_en, uniq = set(), set(), []
for ja, en in sorted(pairs):
    if ja not in seen_ja and en not in seen_en:
        seen_ja.add(ja); seen_en.add(en); uniq.append((ja, en))
random.seed(0)
random.shuffle(uniq)
test, valid, train = uniq[:1000], uniq[1000:1500], uniq[1500:]
print(len(pairs), "→", len(uniq), "対 /", len(train), len(valid), len(test), train[0])

In [ ]:
import sentencepiece as spm

with open("train.txt", "w", encoding="utf-8") as f:
    for ja, en in train:
        f.write(ja + "\n" + en + "\n")
# 日英で語彙を共有する 1 つのサブワードモデル。pad=0, unk=1, bos=2, eos=3
spm.SentencePieceTrainer.train(input="train.txt", model_prefix="spm", vocab_size=16000, character_coverage=0.9995,
                               pad_id=0, unk_id=1, bos_id=2, eos_id=3, input_sentence_size=600000, shuffle_input_sentence=True, minloglevel=2)
sp = spm.SentencePieceProcessor(model_file="spm.model")
print(sp.encode("吾輩は猫である。", out_type=str), sp.encode("I am a cat.", out_type=str))

MAX_TOKENS = 64
def encode(pairs):
    return [(sp.encode(ja)[:MAX_TOKENS - 1] + [3], sp.encode(en)[:MAX_TOKENS - 1] + [3]) for ja, en in pairs]   # 末尾に EOS
train_ids, valid_ids = encode(train), encode(valid)

In [ ]:
import math
import torch
from transformers import MarianConfig, MarianMTModel

device = "cuda" if torch.cuda.is_available() else "cpu"
config = MarianConfig(
    vocab_size=16000, d_model=256, encoder_layers=4, decoder_layers=4, encoder_attention_heads=4, decoder_attention_heads=4,
    encoder_ffn_dim=1024, decoder_ffn_dim=1024, dropout=0.1, max_position_embeddings=128,
    pad_token_id=0, eos_token_id=3, decoder_start_token_id=0, share_encoder_decoder_embeddings=True,
)
model = MarianMTModel(config).to(device)          # 事前学習なし。重みはランダムに初期化される
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M パラメータ")

def batches(data, batch_size, shuffle):
    order = sorted(range(len(data)), key=lambda i: len(data[i][0]))      # 長さの近い文をまとめてパディングを減らす
    chunks = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
    if shuffle:
        random.shuffle(chunks)
    for chunk in chunks:
        src = torch.nn.utils.rnn.pad_sequence([torch.tensor(data[i][0]) for i in chunk], batch_first=True, padding_value=0)
        tgt = torch.nn.utils.rnn.pad_sequence([torch.tensor(data[i][1]) for i in chunk], batch_first=True, padding_value=-100)   # -100 は損失で無視される
        yield src.to(device), tgt.to(device)

In [ ]:
EPOCHS, BATCH, PEAK_LR, WARMUP = 12, 128, 7e-4, 2000
optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, betas=(0.9, 0.98), weight_decay=0.01)
schedule = lambda step: min((step + 1) / WARMUP, math.sqrt(WARMUP / (step + 1)))       # 線形に上げてから、ステップ数の平方根に反比例して下げる
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)
scaler = torch.amp.GradScaler(enabled=device == "cuda")
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)

step = 0
for epoch in range(EPOCHS):
    model.train()
    for src, tgt in batches(train_ids, BATCH, shuffle=True):
        with torch.autocast(device_type=device, enabled=device == "cuda"):
            logits = model(input_ids=src, attention_mask=src != 0, labels=tgt).logits      # labels を渡すと、右に 1 つずらしたものがデコーダの入力になる
            loss = criterion(logits.view(-1, logits.size(-1)), tgt.view(-1))
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        step += 1
        if step % 500 == 0:
            print(f"epoch {epoch + 1} step {step} loss {loss.item():.3f} lr {scheduler.get_last_lr()[0]:.2e}")
    model.eval()
    with torch.no_grad():
        val = [criterion(model(input_ids=s, attention_mask=s != 0, labels=t).logits.flatten(0, 1), t.flatten()).item() for s, t in batches(valid_ids, 256, False)]
    print(f"== epoch {epoch + 1}: 検証損失 {sum(val) / len(val):.3f}")

In [ ]:
import sacrebleu

@torch.no_grad()
def translate(sentences, num_beams=4, batch_size=64):
    model.eval()
    outputs = []
    for i in range(0, len(sentences), batch_size):
        ids = [sp.encode(s)[:MAX_TOKENS - 1] + [3] for s in sentences[i:i + batch_size]]
        src = torch.nn.utils.rnn.pad_sequence([torch.tensor(x) for x in ids], batch_first=True, padding_value=0).to(device)
        out = model.generate(input_ids=src, attention_mask=src != 0, num_beams=num_beams, max_new_tokens=MAX_TOKENS)
        outputs += [sp.decode([t for t in row.tolist() if t > 3]) for row in out]
    return outputs

hyps = translate([ja for ja, _ in test])
print(sacrebleu.corpus_bleu(hyps, [[en for _, en in test]]))
for (ja, en), hyp in list(zip(test, hyps))[:6]:
    print(f"{ja}\n   訳: {hyp}\n   参照: {en}")

- 流れは「対訳データの掃除 → サブワード分割 → モデルの定義 → 学習 → ビームサーチで翻訳 → BLEU で評価」。事前学習済みモデルを使わなくても、十数万文対の短い文なら翻訳らしいものが出てくる。
- モデルは約 1,150 万パラメータの小さな Transformer。RTX A5000 で 12 エポックが約 6 分（T4 ならその 2〜3 倍）。
- 重複を除かずにランダムに分けると、学習データと同じ文がテストに入り、BLEU が約 21 と高めに出た。Tatoeba には同じ文や言い換えの対が多いので、分割の前に重複を除く。
- データの質が結果を大きく左右する。田中コーパスは短く対訳のずれが少ない文が多く、BLEU は 12 エポックで約 18 だった。同じモデルを、字幕のような会話文が多い opus-100（29 万文対）で学習すると、BLEU は 3〜5 にとどまった。
- EDRDG 版は Tatoeba から定期的に更新されるので、実行する日によってデータの件数や BLEU の値が少し変わる。固定の分割で比べたいなら、田中コーパスから 5 万文対を抜き出した [small_parallel_enja](https://github.com/odashi/small_parallel_enja) を使う（英語は小文字化・トークン化済み）。
- 学習率のウォームアップ（最初は小さく、徐々に上げる）がないと、Transformer の学習は発散しやすい。ラベル平滑化と勾配クリッピングも定番の設定。
- 検証損失が下がらなくなったら学習を止める。データが少ないのに層を増やしても、過学習するだけで良くならない。
- 本格的に学習するなら、データを増やし（JParaCrawl など数千万文対）、逆翻訳で単言語データを活用し、fairseq や Marian、OpenNMT などの専用ツールを使う。